# Penalty selection via SDP relaxation

## 1. Il problema della penalty nei QUBO

Consideriamo un problema binario quadratico vincolato:

$ \min_{\mathbf{x}\in\{0,1\}^n} f(\mathbf{x}) = \mathbf{x}^TQ\mathbf{x} $

soggetto a

$ A\mathbf{x}=\mathbf{b}. $

Per poter utilizzare un solver QUBO, i vincoli vengono trasformati in
termini di penalità:

$ \boxed{H_M(\mathbf{x}) = f(\mathbf{x}) + M\|A\mathbf{x}-\mathbf{b}\|^2 } $

dove \(M>0\) è il peso della penalty.

Per una soluzione feasible,

$ A\mathbf{x}-\mathbf{b}=0, $

quindi la penalty scompare.

Per una soluzione infeasible, invece,

$ \|A\mathbf{x}-\mathbf{b}\|^2>0 $

e il valore della funzione obiettivo viene aumentato.

L'obiettivo è scegliere \(M\) sufficientemente grande affinché nessuna
soluzione infeasible possa avere energia minore della soluzione ottima
feasible.

---

## 2. Perché non scegliere semplicemente una penalty enorme?

Una scelta molto conservativa rende sicuramente costosa la violazione dei
vincoli, ma introduce il cosiddetto **Big-M problem**.

Se \(M\) è molto più grande dei coefficienti della funzione obiettivo
originale,

$ f(\mathbf{x}), $

la scala energetica del problema viene dominata dalla penalty.

Questo è particolarmente problematico per solver quantum e
quantum-inspired, perché le differenze energetiche rilevanti associate
alla funzione obiettivo diventano relativamente molto piccole.

L'obiettivo è quindi:

$ \boxed{ \text{scegliere la più piccola penalty che renda la formulazione esatta} } $

ovvero sufficientemente grande da garantire la feasibility, ma non più
grande del necessario.

---

## 3. Quanto deve essere grande \(M\)?

Sia

$ \mathbf{x}_{\mathrm{feas}} $

una qualsiasi soluzione feasible nota.

Il suo costo è:

$ U=f(\mathbf{x}_{\mathrm{feas}}). $

Questa quantità costituisce un **upper bound** sul valore ottimo del
problema vincolato, perché la soluzione ottima non può essere peggiore di
una soluzione feasible già conosciuta:

$ f(\mathbf{x}^\star) \le U. $

Consideriamo ora il problema ottenuto ignorando temporaneamente i vincoli:

$ \min_{\mathbf{x}\in\{0,1\}^n} f(\mathbf{x}). $

Supponiamo di conoscere un lower bound \(L\) tale che

$ L \le \min_{\mathbf{x}\in\{0,1\}^n} f(\mathbf{x}). $

La situazione è quindi:

$ \boxed{ L \le f^\star \le U }$

dove:

- \(U\) proviene da una soluzione feasible;
- \(L\) è un lower bound sulla funzione obiettivo senza i vincoli.

Nel caso peggiore, una soluzione infeasible potrebbe avere costo vicino a
\(L\).

Per essere sicuri che anche questa soluzione diventi peggiore della
soluzione feasible dopo l'aggiunta della penalty, vogliamo:

$ L + M > U. $

Da cui:

$ \boxed{ M > U-L } $

oppure, introducendo un piccolo margine \(\delta>0\),

$ \boxed{ M = U-L+\delta. } $

Questa è l'idea centrale proposta nel paper.

---

## 4. Come ottenere \(U\)?

Non è necessario conoscere la soluzione ottima del problema.

È sufficiente trovare **una qualsiasi buona soluzione feasible**:

$ \mathbf{x}_{\mathrm{feas}}. $

Questa può essere ottenuta, ad esempio, tramite:

- un solver classico con un time limit ridotto;
- una greedy heuristic;
- una soluzione già nota del problema;
- un algoritmo euristico.

Più la soluzione feasible è buona, più piccolo sarà

$ U=f(\mathbf{x}_{\mathrm{feas}}) $

e quindi più stretta sarà la stima della penalty.

---

## 5. Come ottenere \(L\)?

Calcolare esattamente

$ \min_{\mathbf{x}\in\{0,1\}^n} f(\mathbf{x}) $

sarebbe generalmente difficile quanto risolvere il problema originale.

Il paper propone quindi di utilizzare una **Semidefinite Programming
relaxation (SDP)**.

L'idea è rilassare il problema binario quadratico in un problema convesso
più semplice da risolvere.

La SDP considera uno spazio di soluzioni più grande rispetto al problema
binario originale e può quindi ottenere un valore più basso:

$ \boxed{ L_{\mathrm{SDP}} \le f^\star_{\mathrm{unconstrained}}}$

Per questo \(L_{\mathrm{SDP}}\) costituisce un lower bound valido.

La penalty suggerita diventa quindi

$\boxed{ M_{\mathrm{SDP}} = f(\mathbf{x}_{\mathrm{feas}}) - L_{\mathrm{SDP}} + \delta.}$

---

## 6. Interpretazione intuitiva

La procedura può essere vista come il calcolo dell'intervallo

$ \underbrace{L_{\mathrm{SDP}}}_{\text{scenario ottimistico}} \qquad \longrightarrow \qquad \underbrace{f(\mathbf{x}_{\mathrm{feas}})}_{\text{soluzione feasible nota}}. $

La differenza

$ f(\mathbf{x}_{\mathrm{feas}}) - L_{\mathrm{SDP}}$

rappresenta una stima conservativa del **massimo vantaggio che una
soluzione potrebbe ottenere ignorando i vincoli**.

La penalty deve compensare almeno questo vantaggio.

Pertanto:

$ \boxed{ \text{Penalty} \approx \text{costo feasible} - \text{lower bound unconstrained} }$

---

## 7. Workflow

Il metodo può essere sintetizzato come:

$\boxed{ \text{Problema vincolato} \rightarrow \text{soluzione feasible} \rightarrow U } $

e parallelamente

$ \boxed{ \text{funzione obiettivo senza vincoli} \rightarrow \text{SDP relaxation} \rightarrow L_{\mathrm{SDP}} }$

infine

$ \boxed{ M = U-L_{\mathrm{SDP}}+\delta }$

La SDP non viene quindi utilizzata per risolvere direttamente il problema
QUBO finale.

Viene utilizzata come **pre-processing classico** per ottenere un lower
bound e scegliere una penalty più stretta.

Il QUBO così costruito può successivamente essere passato a un solver
classico, quantum-inspired o quantum.

## Teoria della SDP relaxation

Vogliamo ottenere un **lower bound** della funzione obiettivo binaria

$f(\mathbf{x}) = \mathbf{x}^T Q \mathbf{x} + \mathbf{L}^T \mathbf{x}, \qquad \mathbf{x} \in \{0,1\}^n.$

In questa fase ignoriamo temporaneamente i vincoli del problema: l'obiettivo della SDP non è trovare direttamente la soluzione finale, ma ottenere un limite inferiore al miglior valore che la funzione obiettivo potrebbe raggiungere.

---

### 1. Il problema originale è difficile

La funzione contiene termini quadratici $x_i x_j$ e le variabili sono binarie:

$x_i \in \{0,1\}.$

Il problema è quindi combinatorio e, in generale, difficile da risolvere esattamente.

L'idea della SDP relaxation è trasformare questi prodotti in elementi di una matrice.

---

### 2. Lifting del problema

Definiamo il vettore aumentato

$\mathbf{y} = \begin{pmatrix} 1 \\ \mathbf{x} \end{pmatrix}$

e costruiamo la matrice

$Y = \mathbf{y}\mathbf{y}^T.$

Esplicitamente,

$Y = \begin{pmatrix} 1 & \mathbf{x}^T \\ \mathbf{x} & \mathbf{x}\mathbf{x}^T \end{pmatrix}.$

La parte in basso a destra contiene quindi tutti i prodotti pairwise:

$Y_{i+1,j+1} = x_i x_j.$

Per esempio, se

$\mathbf{x} = \begin{pmatrix} 1 \\ 0 \\ 1 \end{pmatrix},$

allora

$\mathbf{x}\mathbf{x}^T = \begin{pmatrix} 1 & 0 & 1 \\ 0 & 0 & 0 \\ 1 & 0 & 1 \end{pmatrix}.$

In questo modo i termini quadratici $x_i x_j$ diventano semplicemente elementi della matrice $Y$.

---

### 3. Anche la funzione obiettivo diventa lineare in $Y$

Definiamo

$\widetilde Q = \begin{pmatrix} 0 & \frac{1}{2}\mathbf{L}^T \\ \frac{1}{2}\mathbf{L} & Q \end{pmatrix}.$

La funzione obiettivo può allora essere riscritta come

$f(\mathbf{x}) = \operatorname{Tr}\left(Y^T \widetilde Q\right).$

Dopo il lifting, l'obiettivo non è più quadratico nelle variabili originarie: è **lineare nella matrice $Y$**.

Questo però non rende ancora il problema semplice, perché dovremmo imporre esattamente

$Y = \mathbf{y}\mathbf{y}^T.$

---

### 4. Perché $Y = \mathbf{y}\mathbf{y}^T$ è ancora difficile?

Una matrice costruita come

$Y = \mathbf{y}\mathbf{y}^T$

possiede due proprietà fondamentali:

$Y \succeq 0$

e

$\operatorname{rank}(Y)=1.$

La prima significa che $Y$ è **positiva semidefinita**.

In altre parole,

$\mathbf{v}^T Y \mathbf{v} \geq 0$

per qualsiasi vettore $\mathbf{v}$.

Infatti, se $Y = \mathbf{y}\mathbf{y}^T$, allora

$\mathbf{v}^T Y \mathbf{v} = \mathbf{v}^T \mathbf{y}\mathbf{y}^T \mathbf{v} = (\mathbf{y}^T \mathbf{v})^2 \geq 0.$

La seconda proprietà,

$\operatorname{rank}(Y)=1,$

è invece problematica: imporre un vincolo di rango 1 rende il problema non convesso.

---

### 5. La relaxation

La SDP relaxation consiste nel **rimuovere il vincolo difficile di rango 1**.

Nel problema esatto avremmo

$Y \succeq 0, \qquad \operatorname{rank}(Y)=1.$

Nella relaxation manteniamo solamente la condizione semidefinita positiva

$Y \succeq 0,$

insieme ad alcuni vincoli lineari che mantengono parte della struttura del problema binario.

Per una variabile binaria vale

$x_i^2 = x_i.$

Poiché gli elementi diagonali della matrice rappresentano

$Y_{i+1,i+1} = x_i^2,$

mentre la prima riga contiene $x_i$, possiamo imporre

$Y_{1,i+1} = Y_{i+1,i+1}.$

Il paper aggiunge inoltre i bounds

$0 \leq Y_{ij} \leq 1.$

La SDP relaxation assume quindi la forma

$\min_Y \operatorname{Tr}(Y^T \widetilde Q)$

soggetta a

$Y \succeq 0,$

$Y_{1i} = Y_{ii},$

$0 \leq Y_{ij} \leq 1.$

---

### 6. Perché si chiama relaxation?

Nel problema originale $Y$ deve necessariamente essere generata da un vettore binario:

$Y = \begin{pmatrix} 1 \\ \mathbf{x} \end{pmatrix}\begin{pmatrix} 1 \\ \mathbf{x} \end{pmatrix}^T.$

Nella SDP permettiamo invece anche matrici positive semidefinite che non possono essere scritte in questa forma con un vettore binario.

Abbiamo quindi **allargato lo spazio delle soluzioni ammesse**:

$\mathcal{F}_{\mathrm{binary}} \subseteq \mathcal{F}_{\mathrm{SDP}}.$

Poiché stiamo minimizzando, avere più soluzioni disponibili può soltanto abbassare il valore ottimo:

$f^\star_{\mathrm{SDP}} \leq f^\star_{\mathrm{binary}}.$

Per questo motivo il risultato della SDP è un **lower bound** valido per il problema binario originale.

---

### 7. Utilizzo per la scelta della penalty

Una volta ottenuti:

- una soluzione feasible $\mathbf{x}_{\mathrm{feas}}$;
- il lower bound $f^\star_{\mathrm{SDP}}$;

possiamo stimare la penalty come

$M_{\mathrm{SDP}} = f(\mathbf{x}_{\mathrm{feas}}) - f^\star_{\mathrm{SDP}} + \delta,$

dove $\delta > 0$ introduce un piccolo margine di sicurezza.

L'interpretazione è che la differenza

$f(\mathbf{x}_{\mathrm{feas}}) - f^\star_{\mathrm{SDP}}$

costituisce una stima conservativa del massimo vantaggio che una soluzione potrebbe ottenere ignorando i vincoli.

La penalty viene quindi scelta in modo da compensare almeno tale vantaggio.

In [ ]:
import sys

sys.path.append("..")

from src.qubo_windfarm_layout.penalties import suggest_cardinality_penalty_from_layout, suggest_spacing_penalty_from_layout

In [3]:
REFERENCE_LAYOUT = "../results/layouts/cpsat_200_3600s.yaml"

lambda_cardinality = (
    suggest_cardinality_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

lambda_spacing = (
    suggest_spacing_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

Reference turbines: 81
Max marginal wake loss: 24,142.767
Safety factor: 1.5
Suggested lambda cardinality: 36,214.151
Reference turbines: 81
Reference pairwise wake objective: 882,505.366
Safety factor: 1.2
Suggested lambda spacing: 1,059,006.439


In [ ]:
print(f"Lambda cardinalty = {lambda_cardinality}")
print(f"Lambda cardinalty = {lambda_spacing}")